# QC-C3-03: Quantum Circuit Optimization Using QSDK
PESU – C-DAC Quantum Hackathon 2026

**Single-file version:** upload only this notebook to Qniverse JupyterLab and run all cells. The first cell unpacks the embedded `qnvkit` package; the second installs target frameworks for verification.

In [ ]:
# Unpacks the embedded qnvkit package next to this notebook (pure Python, needs only numpy).
import base64, io, os, sys, zipfile
_PKG = """
...
"""
_dir = os.path.abspath("qnvkit_pkg")
zipfile.ZipFile(io.BytesIO(base64.b64decode(_PKG.replace("\n", "")))).extractall(_dir)
# Always use THIS embedded copy: put it first and drop any older qnvkit already imported or on the path.
sys.path[:] = [_dir] + [p for p in sys.path if p != _dir]
for _m in [m for m in sys.modules if m == "qnvkit" or m.startswith("qnvkit.")]:
    del sys.modules[_m]
print("qnvkit unpacked to", _dir)

In [ ]:
# Cell: Install execution frameworks for local simulation (Qiskit & Aer)
# Purges cache first and uses --no-cache-dir to prevent container disk exhaustion
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--quiet", "qiskit", "qiskit-aer"], check=False)

In [ ]:
# Which QSDK packages does this Python environment have?
import importlib, importlib.util, subprocess, sys
importlib.invalidate_caches()

print("Python", sys.version.split()[0])
for _n in ("Qniverse", "qniverse"):
    print("importable as %r:" % _n, importlib.util.find_spec(_n) is not None)
_pip = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True).stdout
print([l for l in _pip.splitlines() if "niverse" in l.lower() or "qsdk" in l.lower()])

In [ ]:
import os, sys
import qnvkit, qnvkit.qsdk_adapter as adapter          # must be imported before Qniverse
from qnvkit.optimize import optimize
from qnvkit.metrics import profile
from qnvkit.qsdk_adapter import from_qniverse, to_qniverse
from qnvkit.sim import check_unitary_equivalence

from Qniverse.QniverseCircuit import *
from Qniverse.gates import *
import numpy as np

print("qnvkit loaded from:", os.path.dirname(qnvkit.__file__))
print("QSDK in use:", adapter.qsdk_description())
print("qnvkit optimizer ready!")

In [ ]:
# What the real QSDK does with the constructs this demo uses. Every probe is isolated, so one
# failure does not stop the others. Please keep this output if anything below fails.
import traceback

def _probe(label, build):
    qc = QniverseCircuit()
    q = QuantumRegister(3, 'q'); c = ClassicalRegister(3, 'c')
    qc.add_register(q); qc.add_register(c)
    try:
        build(qc, q, c)
        text = adapter.read_qasm(qc)
        print("[OK]  %s\n%s\n" % (label, "\n".join("      " + l for l in text.splitlines()[4:])))
    except Exception as e:
        print("[ERR] %s -> %s: %s\n" % (label, type(e).__name__, e))

_probe("h, cp, u3, measure", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(cp(0.5), q[0], q[1]),
       qc.add_gate(u3(0.1, 0.2, 0.3), q[1]), qc.add_gate(measure, q[0], c[0])))
_probe("barrier on 2 qubits q[0],q[1]", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0], q[1])))
_probe("barrier on 2 qubits q[0],q[2]", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0], q[2])))
_probe("barrier on 1 qubit", lambda qc, q, c: (qc.add_gate(h, q[0]), qc.add_gate(barrier, q[0])))
_probe("reset + conditional", lambda qc, q, c: (qc.add_gate(measure, q[0], c[0]), qc.add_gate(reset, q[1]),
       qc.add_conditional_gate(c, 1, x, q[1])))
_probe("ccx, swap, crz, measure_all", lambda qc, q, c: (qc.add_gate(ccx, q[0], q[1], q[2]), qc.add_gate(swap, q[0], q[1]),
       qc.add_gate(crz(0.7), q[0], q[1]), qc.measure_all()))

# Source of the SDK's own QASM export around barriers (read-only; for debugging).
try:
    import importlib, inspect
    if not adapter.IS_LIVE_QNIVERSE:
        raise RuntimeError("the official QSDK is not in use, so there is no SDK source to show")
    _m = importlib.import_module(adapter.QSDK_IMPORT_NAME + ".transpile.qasm")
    _lines = inspect.getsource(_m).splitlines()
    _hits = [i for i, l in enumerate(_lines) if "barrier" in l.lower()]
    print("--- SDK source lines mentioning 'barrier' (with context) ---")
    _shown = set()
    for i in _hits:
        for j in range(max(0, i - 6), min(len(_lines), i + 22)):
            if j not in _shown:
                _shown.add(j); print("%4d  %s" % (j + 1, _lines[j]))
        print("      ...")
except Exception as e:
    print("source not available:", e)

## 1. Benchmark Circuit 1: Multi-Qubit Circuit with All Redundancy Classes
We construct a 3-qubit circuit containing:
- Self-inverse pairs ($H \cdot H, X \cdot X$)
- Consecutive identical $CX$ pairs ($CX(0, 1) \cdot CX(0, 1)$)
- Consecutive adjoint pairs ($S \cdot S^\dagger$)
- Continuous rotation merging ($R_z(0.3) + R_z(0.5) \to R_z(0.8)$)
- An intentional barrier protecting two $Z$ gates from cancelling (Negative Test)

In [ ]:
qc = QniverseCircuit()
q = QuantumRegister(3, 'q')
c = ClassicalRegister(3, 'c')
qc.add_register(q)
qc.add_register(c)

# 1. Self-inverse H-H and X-X pairs
qc.add_gate(h, q[0])
qc.add_gate(h, q[0])          # Cancels with H(0)
qc.add_gate(x, q[1])
qc.add_gate(x, q[1])          # Cancels with X(1)

# 2. CX-CX cancellation
qc.add_gate(cx, q[0], q[1])
qc.add_gate(cx, q[0], q[1])   # Cancels with CX(0,1)

# 3. Inverse pair S-Sdg
qc.add_gate(s, q[2])
qc.add_gate(sdg, q[2])        # Cancels with S(2)

# 4. Continuous rotation merging
qc.add_gate(rz(0.3), q[0])
qc.add_gate(rz(0.5), q[0])    # Merges to Rz(0.8)

# 5. Intentional pair PROTECTED by a barrier (Negative Test)
qc.add_gate(z, q[1])
qc.add_gate(barrier, q[0], q[2])  # Timing boundary
qc.add_gate(z, q[1])          # MUST NOT cancel across barrier!

# 6. Computational gate and measurements
qc.add_gate(cx, q[1], q[2])
for i in range(3):
    qc.add_gate(measure, q[i], c[i])

print('Unoptimized OpenQASM 2.0:')
print(qc.qasm())

print('\n--- Unoptimized Circuit Diagram (qc.draw()) ---')
qc.draw()


## 2. Circuit Profiling (Before Optimization)
Calculate gate count, breakdown by gate type, and earliest-available-layer scheduled depth using `qnvkit.metrics`.

In [ ]:
before_prof = profile(qc)
print(before_prof.summary())

## 3. Mandatory Optimization Passes
Applies the mandatory pass set: `remove_identity`, `cancel_self_inverse`, `cancel_cx`.

In [ ]:
res_mandatory = optimize(qc)      # default: remove_identity, cancel_self_inverse, cancel_cx (one round)
print(res_mandatory.summary())

## 4. Full Optimization Pipeline (With Rotation Merging & Fixed Point)
Merges $R_z(0.3) + R_z(0.5) \to R_z(0.8)$, cancels adjoint pairs ($S \cdot S^\dagger$), and loops to fixed point.

In [ ]:
all_passes = [
    'remove_identity',
    'cancel_self_inverse',
    'cancel_cx',
    'cancel_extended_self_inverse',
    'cancel_inverse_pairs',
    'merge_rotations',
]                                  # same as passes="all"
res_full = optimize(qc, passes=all_passes, fixed_point=True)
print(res_full.summary())
print('\nOptimized OpenQASM 2.0:')
print(res_full.circuit.qasm())

print('\n--- Optimized Circuit Diagram (res_full.circuit.draw()) ---')
res_full.circuit.draw()

## 5. Negative Test: Barrier Timing Protection
A critical safety requirement: the optimizer must **never** cancel gates separated by a barrier.

In [ ]:
z_count = res_full.after.gate_counts.get("z", 0)
print(f"Remaining Z gates: {z_count} (Expected: 2, protected by barrier)")
assert z_count == 2, "Security guardrail failed: barrier did not protect gates!"
print("[PASSED] Barrier strictly preserved intentional timing semantics.")

## 6. Comprehensive Test Suite: Discrete Pass Verifications
We rigorously test every single rewrite rule in isolation to demonstrate complete test case coverage.

In [ ]:
# Test A: Identity and Zero Rotations
q_test = QuantumRegister(2, 'q')
qc_a = QniverseCircuit()
qc_a.add_register(q_test)
qc_a.add_gate(id, q_test[0])
qc_a.add_gate(rx(0.0), q_test[1])
qc_a.add_gate(rz(2.0 * np.pi), q_test[1])
res_a = optimize(qc_a, passes=["remove_identity"])
print(f"[Test A: remove_identity] 3 gates -> {res_a.after.total_gates} gates (Expected: 0) | PASS: {res_a.after.total_gates == 0}")

# Test B: Multi-Qubit Extended Self-Inverse (CZ-CZ, SWAP-SWAP, CH-CH)
qc_b = QniverseCircuit()
qc_b.add_register(q_test)
qc_b.add_gate(cz, q_test[0], q_test[1])
qc_b.add_gate(cz, q_test[0], q_test[1])
qc_b.add_gate(swap, q_test[0], q_test[1])
qc_b.add_gate(swap, q_test[0], q_test[1])
res_b = optimize(qc_b, passes=["cancel_extended_self_inverse"])
print(f"[Test B: cancel_extended] 4 gates -> {res_b.after.total_gates} gates (Expected: 0) | PASS: {res_b.after.total_gates == 0}")

# Test C: Adjoint Pairs (T-Tdg, CS-CSdg)
qc_c = QniverseCircuit()
qc_c.add_register(q_test)
qc_c.add_gate(t, q_test[0])
qc_c.add_gate(tdg, q_test[0])
qc_c.add_gate(cs, q_test[0], q_test[1])
qc_c.add_gate(csdg, q_test[0], q_test[1])
res_c = optimize(qc_c, passes=["cancel_inverse_pairs"])
print(f"[Test C: cancel_inverse]  4 gates -> {res_c.after.total_gates} gates (Expected: 0) | PASS: {res_c.after.total_gates == 0}")

# Test D: Rotation Merging to 2*pi (Cancellation)
qc_d = QniverseCircuit()
qc_d.add_register(q_test)
qc_d.add_gate(rz(np.pi), q_test[0])
qc_d.add_gate(rz(np.pi), q_test[0])  # Merges to 2*pi -> drops completely!
res_d = optimize(qc_d, passes=["merge_rotations", "remove_identity"])
print(f"[Test D: merge_to_2pi]    2 gates -> {res_d.after.total_gates} gates (Expected: 0) | PASS: {res_d.after.total_gates == 0}")

# Test E: Wire Commutation Across Disjoint Qubits
qc_e = QniverseCircuit()
qc_e.add_register(q_test)
# X on wire 0, H on wire 1, X on wire 0, H on wire 1
qc_e.add_gate(x, q_test[0])
qc_e.add_gate(h, q_test[1])
qc_e.add_gate(x, q_test[0])
qc_e.add_gate(h, q_test[1])
res_e = optimize(qc_e, passes=["cancel_self_inverse"])
print(f"[Test E: commutation]     4 gates -> {res_e.after.total_gates} gates (Expected: 0) | PASS: {res_e.after.total_gates == 0}")

# Test F: Asymmetric Control/Target CX Protection
qc_f = QniverseCircuit()
qc_f.add_register(q_test)
qc_f.add_gate(cx, q_test[0], q_test[1])
qc_f.add_gate(cx, q_test[1], q_test[0])  # Reversed! MUST NOT CANCEL!
res_f = optimize(qc_f, passes=["cancel_cx"])
print(f"[Test F: reverse_cx]      2 gates -> {res_f.after.total_gates} gates (Expected: 2) | PASS: {res_f.after.total_gates == 2}")

## 7. Algorithmic Benchmark: 4-Qubit Circuit Optimization
Optimizes a larger 4-qubit circuit with layered entanglement, rotation merges, and inverse cancellations.

In [ ]:
q4 = QuantumRegister(4, 'q')
c4 = ClassicalRegister(4, 'c')
qc_bench = QniverseCircuit()
qc_bench.add_register(q4)
qc_bench.add_register(c4)

# Entanglement layer with cancellations
for i in range(4):
    qc_bench.add_gate(h, q4[i])
    qc_bench.add_gate(h, q4[i])  # Cancel
    qc_bench.add_gate(x, q4[i])

qc_bench.add_gate(cx, q4[0], q4[1])
qc_bench.add_gate(cx, q4[0], q4[1])  # Cancel
qc_bench.add_gate(cz, q4[2], q4[3])
qc_bench.add_gate(cz, q4[2], q4[3])  # Cancel

# Rotations
qc_bench.add_gate(rz(0.4), q4[0])
qc_bench.add_gate(rz(0.6), q4[0])    # Merge to Rz(1.0)
qc_bench.add_gate(rx(0.2), q4[1])
qc_bench.add_gate(rx(0.3), q4[1])    # Merge to Rx(0.5)

# Measurements
for i in range(4):
    qc_bench.add_gate(measure, q4[i], c4[i])

bench_res = optimize(qc_bench, passes=all_passes, fixed_point=True)
print('=== 4-Qubit Algorithmic Benchmark ===')
print(bench_res.summary())
print()

print('--- 4-Qubit Benchmark Circuit (Before: qc_bench.draw()) ---')
qc_bench.draw()
print('\n--- 4-Qubit Benchmark Circuit (After: bench_res.circuit.draw()) ---')
bench_res.circuit.draw()

## 8. Mathematical Equivalence & Immutability Verification
Every number below is computed in this cell:
1. Process fidelity $|\mathrm{Tr}(U_{orig}^\dagger U_{opt})| / 2^n$ between the original and the optimized circuit (1.0 = equal up to global phase).
2. The original circuit object exports the same OpenQASM before and after optimization (the input is never modified).
3. The optimizer's own verdict, `result.equivalence_verified`.

In [ ]:
from qnvkit.sim import unitary_fidelity

qasm_before = qc.qasm()
optimize(qc, passes=all_passes, fixed_point=True)          # run again to show the input is untouched
unmodified = qc.qasm() == qasm_before

fid_3q = unitary_fidelity(from_qniverse(qc), res_full.ir)
fid_4q = unitary_fidelity(from_qniverse(qc_bench), bench_res.ir)

print('=' * 80)
print('             MATHEMATICAL EQUIVALENCE & IMMUTABILITY VERIFICATION')
print('=' * 80)
print(f'3-Qubit Circuit Unitary Fidelity:  {fid_3q:.8f} | optimizer verdict: {res_full.equivalence_verified}')
print(f'4-Qubit Circuit Unitary Fidelity:  {fid_4q:.8f} | optimizer verdict: {bench_res.equivalence_verified}')
print(f'Source Circuit Unmodified:         {unmodified}')
print(f'Timing Boundary (Barrier) Honored: {res_full.after.gate_counts.get("z", 0) == 2} (both Z gates kept)')
print('=' * 80)
assert fid_3q > 1 - 1e-9 and fid_4q > 1 - 1e-9 and unmodified

print('\n--- Visual Circuit Comparison: Unoptimized (qc.draw()) ---')
qc.draw()
print('\n--- Visual Circuit Comparison: Optimized (res_full.circuit.draw()) ---')
res_full.circuit.draw()

## 9. Execution: unoptimized vs optimized measurement distributions
Both circuits are run for 1024 shots and their count distributions are compared (total variation distance).
With a Qniverse API key in `QNIVERSE_API_KEY` they are submitted as jobs to the Qniverse backend
(CPU / QACC_Cluster / qasm_simulator); without a key, or if the submission fails, they are simulated locally,
and the table says which one happened.

In [ ]:
token = os.environ.get('QNIVERSE_API_KEY', '')  # <- Paste your Qniverse API key here (keep out of git)
SHOTS = 1024

def _run_on_qniverse(circ, job_name):
    """Submits one job through the QSDK and waits for it. Returns (counts, where_it_ran)."""
    import importlib, pkgutil, time
    import Qniverse
    from Qniverse.backend import Backend

    def _find(name):  # the job functions live in different modules across QSDK versions
        for m in pkgutil.walk_packages(Qniverse.__path__, Qniverse.__name__ + '.'):
            try:
                mod = importlib.import_module(m.name)
            except Exception:
                continue
            if callable(getattr(mod, name, None)):
                return getattr(mod, name)
        raise ImportError(f'{name} not found in QSDK')

    run = _find('run'); fetch_job_result = _find('fetch_job_result'); job_status = _find('job_status')
    backend = Backend()
    backend.processor('CPU')
    backend.hpc('QACC_Cluster')
    backend.simulator('qasm_simulator')

    job = run(circ, backend, SHOTS, token=token, job_name=job_name)
    t0 = time.time()
    status = 'SUBMITTED'
    while time.time() - t0 < 300:
        status = str(job_status(token, job)).upper()
        if 'COMPLETED' in status or 'FAILED' in status:
            break
        time.sleep(2)
    if 'COMPLETED' not in status:
        raise RuntimeError(f'job {job} ended with status {status}')
    raw = fetch_job_result(token, job)
    counts = raw.get('counts') or raw.get('probability') or raw
    return counts, f'{adapter.qsdk_description()} job {job}'

def _run(circ, job_name):
    if token.strip():
        try:
            return _run_on_qniverse(circ, job_name)
        except Exception as e:
            print(f'  [FALLBACK] Qniverse submission failed ({e}); using local simulation.')
    from qnvkit.viz import simulate_circuit
    return simulate_circuit(circ, shots=SHOTS)

runs = {label: _run(circ, 'c3' + label) for label, circ in (('unoptimized', qc), ('optimized', res_full.circuit))}

def _distribution(counts):
    numeric = {str(k).replace(' ', ''): float(v) for k, v in counts.items() if isinstance(v, (int, float))}
    total = sum(numeric.values()) or 1.0
    return {k: v / total for k, v in numeric.items()}

from qnvkit.sim import total_variation_distance
tvd = total_variation_distance(_distribution(runs['unoptimized'][0]), _distribution(runs['optimized'][0]))

print('=' * 80)
print(f'             {SHOTS}-SHOT MEASUREMENT & PERFORMANCE COMPARISON')
print('=' * 80)
print(f"{'Metric':<28} | {'Unoptimized Circuit':<22} | {'Optimized Circuit':<22}")
print('-' * 80)
print(f"{'Unitary Gate Count':<28} | {res_full.before.total_gates:<22} | {res_full.after.total_gates:<22}")
print(f"{'Circuit Scheduled Depth':<28} | {res_full.before.depth:<22} | {res_full.after.depth:<22}")
print(f"{'Gate Count Reduction':<28} | {'-':<22} | {str(res_full.reduction_percentage) + '%':<22}")
print(f"{'Depth Reduction':<28} | {'-':<22} | {str(res_full.depth_reduction_percentage) + '%':<22}")
print('-' * 80)
for label, (counts, where) in runs.items():
    print(f'  * {label.capitalize():<12} ran on: {where}')
    print(f'    {"":<12} counts: {dict(sorted(counts.items()))}')
print('-' * 80)
print(f'Total variation distance between the two distributions: {tvd:.4f} (sampling noise only if < 0.05)')
print('=' * 80)
print(f'VERDICT: {"matching" if tvd < 0.05 else "DIFFERENT"} measurement distributions '
      f'with {res_full.reduction_percentage}% fewer gates.')